# TerraTree (Kali track) — Step 2: Real Ground-Truth Labels + Feature Extraction

This notebook builds TWO genuine, defensible targets from your real field
data — not proxy labels this time:

1. **Dominant plant family per plot** (classification) — species-level
   was checked and rejected: 265 species across only 56 plots with ~42
   species co-occurring per plot gives no clean per-pixel signal. Family
   level is far more balanced (Lauraceae dominates 19/56 plots, Rubiaceae
   10, Fabaceae 8) and is genuine, ground-truth-derived information.
2. **Species richness + Shannon diversity index per plot** (regression) —
   a legitimate, well-established remote-sensing research task, and a
   better fit for what this dataset actually is (a floristic survey).

Honest caveat carried through both: n=56 plots total. This is small.
Results should be reported as indicative, not a robust deployable model —
say this plainly in your report rather than let a panelist find it unstated.

In [ ]:
!pip install geemap -q

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='terratree')

import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()  # upload KTR_actual_ground_truth.csv
gt_df = pd.read_csv('KTR_actual_ground_truth.csv')
print(gt_df.shape)

## Build the two targets, per plot

In [ ]:
# --- Target 1: dominant family, grouped ---
dom_family = gt_df.groupby('plot_id')['family'].agg(
    lambda x: x.value_counts().idxmax() if x.notna().any() else 'Unknown'
)

# Keep families that dominate at least 3 plots as their own class; group the rest
family_counts = dom_family.value_counts()
KEEP_FAMILIES = family_counts[family_counts >= 3].index.tolist()
print("Named family classes (>=3 plots):", KEEP_FAMILIES)

dom_family_grouped = dom_family.apply(lambda f: f if f in KEEP_FAMILIES else 'Other family')
print("\nFinal class distribution:")
print(dom_family_grouped.value_counts())

# --- Target 2: species richness + Shannon diversity, per plot ---
def shannon_index(species_series):
    counts = species_series.value_counts()
    proportions = counts / counts.sum()
    return -(proportions * np.log(proportions)).sum()

richness = gt_df.groupby('plot_id')['species'].nunique()
shannon = gt_df.groupby('plot_id')['species'].apply(shannon_index)

# --- Combine into one per-plot table ---
plot_coords = gt_df.groupby('plot_id')[['decimalLatitude', 'decimalLongitude']].first()

labels_df = plot_coords.copy()
labels_df['dominant_family'] = dom_family_grouped
labels_df['species_richness'] = richness
labels_df['shannon_index'] = shannon
labels_df = labels_df.reset_index()

print(f"\n{len(labels_df)} plots with labels built.")
labels_df.head()

## Rebuild the Sentinel-1/2 feature stack (same as notebook 01)

In [ ]:
import requests

QUERY_CANDIDATES = [
    "Kali Tiger Reserve, Karnataka, India",
    "Anshi Dandeli Tiger Reserve",
    "Dandeli Wildlife Sanctuary",
    "Kali Tiger Reserve",
]
aoi = None
for query in QUERY_CANDIDATES:
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": query, "format": "geojson", "polygon_geojson": 1, "limit": 1},
        headers={"User-Agent": "terratree-major-project"}
    )
    features = resp.json().get('features', [])
    if features:
        aoi = ee.Geometry(features[0]['geometry'])
        break
if aoi is None:
    aoi = ee.Geometry.Rectangle([74.20, 14.80, 74.75, 15.55])

START_DATE, END_DATE, MAX_CLOUD_PCT = '2023-01-01', '2025-12-31', 30

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi).filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT))
    .map(mask_s2_clouds)
)
s2_median = s2_collection.median().clip(aoi)
NEEDED_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
s2_median = s2_median.select(NEEDED_BANDS)

def add_s2_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = img.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndvire = img.normalizedDifference(['B8', 'B5']).rename('NDVIre')
    ndi45 = img.normalizedDifference(['B5', 'B4']).rename('NDI45')
    ndre1 = img.normalizedDifference(['B6', 'B5']).rename('NDre1')
    return img.addBands([ndvi, ndwi, ndvire, ndi45, ndre1])

s2_features = add_s2_indices(s2_median)

s1_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi).filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
)
s1_median = s1_collection.select(['VV', 'VH']).median().toFloat().clip(aoi)
ratio = s1_median.select('VH').divide(s1_median.select('VV')).rename('VH_VV_ratio')
s1_features = s1_median.addBands(ratio)

feature_stack = s2_features.addBands(s1_features)
print("Feature stack bands:", feature_stack.bandNames().getInfo())

## Extract features at each of the 56 plot locations

Only 56 points — this is cheap and fast, nothing like the full-region
sampling that caused memory/timeout issues on the Sundarbans track.
Each plot is buffered by its recorded GPS uncertainty (100m) and we take
the MEAN feature values within that buffer, since the exact plot position
within that 100m circle isn't known precisely.

In [ ]:
BUFFER_METERS = 100

ee_features_list = []
for _, row in labels_df.iterrows():
    point = ee.Geometry.Point([row['decimalLongitude'], row['decimalLatitude']]).buffer(BUFFER_METERS)
    ee_features_list.append(ee.Feature(point, {'plot_id': row['plot_id']}))

plots_fc = ee.FeatureCollection(ee_features_list)

sampled = feature_stack.reduceRegions(
    collection=plots_fc,
    reducer=ee.Reducer.mean(),
    scale=10
)

sampled_list = sampled.getInfo()['features']
sampled_rows = [f['properties'] for f in sampled_list]
sampled_df = pd.DataFrame(sampled_rows)
print(sampled_df.shape)
sampled_df.head()

## Merge features with labels, check for missing data

In [ ]:
training_table = labels_df.merge(sampled_df, on='plot_id', how='inner')
print(f"Merged table: {training_table.shape}")

missing = training_table.isna().sum()
print("\nMissing values per column:")
print(missing[missing > 0])

training_table = training_table.dropna()
print(f"\nAfter dropping rows with missing values: {training_table.shape}")

training_table.to_csv('kali_training_table.csv', index=False)
files.download('kali_training_table.csv')
print("Saved and downloading kali_training_table.csv")

## Next steps
- [ ] Confirm the merged table has close to 56 rows (some may drop if a plot's buffer fell entirely outside imagery coverage)
- [ ] Check `dominant_family` class counts look reasonable (matches the earlier analysis)
- [ ] Move to `03_model_training.ipynb` — trains BOTH the family classifier and the biodiversity regressor from this one table, using Leave-One-Out cross-validation (appropriate given n=56, not a standard train/test split)